# HuBMAP Vasculature — InternImage-T Cascade Inference (Kaggle)

mmdet v3 + InternImage-T(DCNv3_pytorch) + Cascade Mask R-CNN 單模推理 + 6 視角 TTA + morph 後處理。

## 使用流程
1. 把 `hubmap-internimage-cascade` Kaggle dataset 加進 notebook 的 input（包含 code/ configs/ weights/ wheels/）
2. 把 HuBMAP 比賽資料 `hubmap-hacking-the-human-vasculature` 加進 input
3. **Save & Run All（網路 ON）** 跑通驗證
4. 正式提交時 Settings → **Internet OFF** → Save & Run All（離線 wheel 安裝，等價於 ON）

## 警告
- 提交時 **必須關網**（HuBMAP 競賽規定）。所有依賴都已預裝在 dataset 的 `wheels/`，全程 `--no-index` 安裝。
- 若 Kaggle 升 torch / Python，需要重跑 `package_for_kaggle.sh --torch-version ... --cu-version ... --python-version ...` 重抓 wheel。


In [ ]:
# ===== Cell 2: 偵測 Kaggle dataset 真實路徑 + symlink 成扁平 inputs/<slug>/ =====
# Kaggle 新版掛載格式：
#   /kaggle/input/datasets/<owner>/<slug>/   ← user dataset
#   /kaggle/input/competitions/<slug>/        ← competition data
#   /kaggle/input/<slug>/                     ← 舊版扁平掛載（forward-compat）
# 我們統一 symlink 到 /kaggle/working/inputs/<slug>/，後面 cell 用扁平 path。
import os
import glob

SYMLINK_DIR = '/kaggle/working/inputs'
os.makedirs(SYMLINK_DIR, exist_ok=True)

SLUG_TO_REAL = {}
for owner_dir in sorted(glob.glob('/kaggle/input/datasets/*')):
    if os.path.isdir(owner_dir):
        for ds in sorted(glob.glob(os.path.join(owner_dir, '*'))):
            if os.path.isdir(ds):
                SLUG_TO_REAL[os.path.basename(ds)] = ds
for ds in sorted(glob.glob('/kaggle/input/competitions/*')):
    if os.path.isdir(ds):
        SLUG_TO_REAL[os.path.basename(ds)] = ds
for ds in sorted(glob.glob('/kaggle/input/*')):
    name = os.path.basename(ds)
    if name in ('datasets', 'competitions'):
        continue
    if os.path.isdir(ds) and name not in SLUG_TO_REAL:
        SLUG_TO_REAL[name] = ds

for slug, real in SLUG_TO_REAL.items():
    link = os.path.join(SYMLINK_DIR, slug)
    if not os.path.exists(link):
        os.symlink(real, link)

print('偵測到的 datasets:')
for slug in sorted(SLUG_TO_REAL):
    print(f'  {slug:50s} -> {SLUG_TO_REAL[slug]}')

# 確認關鍵兩個 dataset 都在
PKG_SLUG = 'hubmap-internimage-cascade'      # ← 自己的權重 + wheels
COMP_SLUG = 'hubmap-hacking-the-human-vasculature'  # ← 比賽資料

assert PKG_SLUG in SLUG_TO_REAL, f'缺少 {PKG_SLUG} dataset，請把它 add to notebook input'
assert COMP_SLUG in SLUG_TO_REAL, f'缺少 {COMP_SLUG} 比賽資料，請 add the competition data'

PKG_DIR = f'/kaggle/working/inputs/{PKG_SLUG}'
COMP_DIR = f'/kaggle/working/inputs/{COMP_SLUG}'
print(f'\nPKG_DIR  = {PKG_DIR}')
print(f'COMP_DIR = {COMP_DIR}')


In [ ]:
# ===== Cell 3: 離線安裝 mmdet/mmcv 全家桶（--no-index）=====
# wheel 已預先放在 PKG_DIR/wheels/，pip 自動處理依賴順序。
import subprocess
WHEEL_DIR = f'{PKG_DIR}/wheels'
wheels = sorted(__import__('glob').glob(f'{WHEEL_DIR}/*.whl'))
print(f'發現 {len(wheels)} 個 wheel:')
for w in wheels:
    print(f'  - {os.path.basename(w)}')

cmd = ['pip', 'install', '-q', '--no-index', '--find-links', WHEEL_DIR] + wheels
print(f'\n執行: pip install -q --no-index --find-links {WHEEL_DIR} <{len(wheels)} wheels>')
ret = subprocess.run(cmd, capture_output=True, text=True)
print(ret.stdout[-2000:] if ret.stdout else '')
if ret.returncode != 0:
    print('STDERR:'); print(ret.stderr[-2000:])
    raise RuntimeError(f'pip install 失敗 (exit {ret.returncode})')
print('✓ 全部 wheel 安裝完成')

# 立刻驗證
import importlib
for mod in ['mmengine', 'mmcv', 'mmdet', 'albumentations', 'pycocotools']:
    m = importlib.import_module(mod)
    print(f'  {mod:20s} = {getattr(m, "__version__", "?")}')


In [ ]:
# ===== Cell 4: torch.load monkey-patch（與本地 train.py 同步）=====
# 為什麼需要：torch 2.6+ 把 torch.load(weights_only=True) 改成預設，
# 但 mmengine 0.10.x 的 load_from_local 沒傳 weights_only=False，會擋下
# ckpt 裡的 HistoryBuffer 等 mmengine 自家物件。我們載自己訓的 ckpt 是 trusted。
import torch

if not getattr(torch.load, '_hubmap_patched', False):
    _orig_torch_load = torch.load
    def _torch_load_patched(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return _orig_torch_load(*args, **kwargs)
    _torch_load_patched._hubmap_patched = True
    torch.load = _torch_load_patched
    print('✓ torch.load patched (weights_only=False 預設)')
else:
    print('已 patched 過，跳過')


In [ ]:
# ===== Cell 5: 把 code/（含 models/）複製到 /kaggle/working/ 並加入 sys.path =====
# 為什麼複製不直接 sys.path inputs/：
#   - /kaggle/working/ 可寫，避免將來 mmdet/cython 想寫 __pycache__ 失敗
#   - subprocess 在 /kaggle/working/ 起來時找得到 models/
import sys
import shutil

WORK_CODE = '/kaggle/working/code'
if os.path.exists(WORK_CODE):
    shutil.rmtree(WORK_CODE)
shutil.copytree(f'{PKG_DIR}/code', WORK_CODE)

if WORK_CODE not in sys.path:
    sys.path.insert(0, WORK_CODE)

print(f'code/ 已複製到 {WORK_CODE}')
print('內容:')
for root, dirs, files in os.walk(WORK_CODE):
    rel = os.path.relpath(root, WORK_CODE)
    for f in sorted(files):
        if not f.endswith(('.pyc',)):
            print(f'  {os.path.join(rel, f) if rel != "." else f}')


In [ ]:
# ===== Cell 6: 子程序跑 kaggle_infer.py（隔離 Python state，避免污染 notebook kernel）=====
import subprocess

# 自動找 best ckpt
ckpts = sorted(__import__('glob').glob(f'{PKG_DIR}/weights/best_coco_segm_mAP_epoch_*.pth'))
assert ckpts, f'{PKG_DIR}/weights/ 沒有 best ckpt'
CKPT = ckpts[-1]
print(f'使用 ckpt: {os.path.basename(CKPT)}')

CONFIG = f'{PKG_DIR}/configs/stage2_dataset1_finetune.py'
IMG_DIR = f'{COMP_DIR}/test'
OUT_CSV = '/kaggle/working/submission.csv'

cmd = [
    'python', f'{WORK_CODE}/kaggle_infer.py',
    '--config', CONFIG,
    '--checkpoint', CKPT,
    '--img-dir', IMG_DIR,
    '--out', OUT_CSV,
    '--no-compile',          # Cascade RoI head 在 T4 上 torch.compile 易 graph break
    '--score-thr', '0.3',
]
print('執行:', ' '.join(cmd))
print('-' * 60)

# 即時把子程序 stdout 串到 notebook（不要 capture_output 才能看 tqdm/print）
ret = subprocess.run(cmd, cwd=WORK_CODE)
if ret.returncode != 0:
    raise RuntimeError(f'kaggle_infer.py exit {ret.returncode}')

print('-' * 60)
print(f'✓ 推理完成 → {OUT_CSV}')


In [ ]:
# ===== Cell 7: 檢查 submission.csv 格式 =====
import pandas as pd

OUT_CSV = '/kaggle/working/submission.csv'
df = pd.read_csv(OUT_CSV)
print(f'submission.csv: {len(df)} 列')
print()
print('欄位:', list(df.columns))
print()
print('前 5 列（pred_string 截短）:')
preview = df.copy()
preview['prediction_string'] = preview['prediction_string'].astype(str).str[:80] + '...'
print(preview.head().to_string(index=False))

# 驗證格式：欄位齊全、id 不重複、prediction_string 不為空（至少有 label=0 的開頭）
assert set(df.columns) == {'id', 'height', 'width', 'prediction_string'}, '欄位不對'
assert df['id'].is_unique, 'id 有重複'
n_empty = df['prediction_string'].fillna('').eq('').sum()
print(f'\n空 prediction_string 列數: {n_empty}（這些 tile 沒預測到 blood_vessel）')
